In [ ]:
import pandas as pd
from IPython.display import display

from study.data import canonical_stays, record_cohort_step
from study.io import DATA, RESOURCES, WORK, write_table
from study.mimic import calculate_age, extract_mimic, mimiciv_heights
from study.readers import read_table

In [ ]:
MIMICIII, MIMICIV = DATA / 'mimic_iii', DATA / 'mimic_iv'
MIMICIII_FLOW_IDS = {'patients': 'SUBJECT_ID', 'hospital_admissions': 'HADM_ID', 'icu_stays': 'ICUSTAY_ID'}
MIMICIV_FLOW_IDS = {'patients': 'SUBJECT_ID', 'hospital_admissions': 'HADM_ID', 'icu_stays': 'STAY_ID'}
COHORT_ROLES = {'carevue': 'source', 'metavision': 'source', 'mimiciv': 'test'}

In [ ]:
# MIMIC III
stays = read_table(MIMICIII, 'ICUSTAYS', parse_dates = ['INTIME', 'OUTTIME'])
patients = read_table(MIMICIII, 'PATIENTS', parse_dates = ['DOB', 'DOD'])
admissions = read_table(MIMICIII, 'ADMISSIONS', parse_dates = ['ADMITTIME', 'DISCHTIME', 'DEATHTIME'])

carevue_patients = set(stays.loc[stays.DBSOURCE.str.lower().eq('carevue'), 'SUBJECT_ID'])
cohorts = {}
cohort_flows = []
flow_ids = MIMICIII_FLOW_IDS

for name in ('carevue', 'metavision'):
    # Select stays from the current recording system
    era = record_cohort_step(
        cohort_flows,
        name,
        'recording_system',
        stays,
        stays.loc[stays.DBSOURCE.str.lower().eq(name)].copy(),
        flow_ids,
    )

    # Keep cohorts disjoint at the patient level
    if name == 'metavision':
        era = record_cohort_step(
            cohort_flows,
            name,
            'exclude_carevue_patients',
            era,
            era.loc[~era.SUBJECT_ID.isin(carevue_patients)],
            flow_ids,
        )

    # Select first stay before applying age or duration restrictions
    first = record_cohort_step(
        cohort_flows,
        name,
        'first_icu_stay',
        era,
        era.sort_values(['SUBJECT_ID', 'INTIME', 'ICUSTAY_ID']).drop_duplicates('SUBJECT_ID'),
        flow_ids,
    )

    # Link demographics and admission outcomes
    cohort_rows = record_cohort_step(
        cohort_flows,
        name,
        'patient_record',
        first,
        first.merge(patients[['SUBJECT_ID', 'GENDER', 'DOB', 'DOD']], on = 'SUBJECT_ID'),
        flow_ids,
    )
    cohort_rows = record_cohort_step(
        cohort_flows,
        name,
        'admission_record',
        cohort_rows,
        cohort_rows.merge(
            admissions[['HADM_ID', 'ADMITTIME', 'DISCHTIME', 'DEATHTIME', 'HOSPITAL_EXPIRE_FLAG']], on = 'HADM_ID'
        ),
        flow_ids,
    )

    # Retain adults whose ICU stay lasts at least 48 hours
    cohort_rows['AGE_AT_ICU_ADMISSION'] = calculate_age(cohort_rows.DOB, cohort_rows.INTIME)
    cohort_rows = record_cohort_step(
        cohort_flows,
        name,
        'age_at_least_18',
        cohort_rows,
        cohort_rows.loc[cohort_rows.AGE_AT_ICU_ADMISSION.ge(18)],
        flow_ids,
    )
    cohort_rows = record_cohort_step(
        cohort_flows,
        name,
        'icu_stay_at_least_48_hours',
        cohort_rows,
        cohort_rows.loc[cohort_rows.LOS.ge(2)].copy(),
        flow_ids,
    )

    # Store outcome and identifiers
    cohort_rows['IN_HOSPITAL_MORTALITY'] = cohort_rows.HOSPITAL_EXPIRE_FLAG.astype(int)
    cohort_rows['RecordID'] = cohort_rows.ICUSTAY_ID.astype(int)
    cohorts[name] = cohort_rows.drop(columns = ['DOB', 'DOD']).sort_values('RecordID')

In [ ]:
# MIMIC IV
stays = read_table(MIMICIV, 'ICUSTAYS', parse_dates = ['intime', 'outtime']).rename(columns = str.upper)
patients = read_table(MIMICIV, 'PATIENTS').rename(columns = str.upper)
admissions = read_table(MIMICIV, 'ADMISSIONS', parse_dates = ['admittime', 'dischtime', 'deathtime']).rename(
    columns = str.upper
)

# Link patient records before applying the year restriction
flow_ids = MIMICIV_FLOW_IDS
candidates = record_cohort_step(
    cohort_flows, 'mimiciv', 'patient_record', stays, stays.merge(patients, on = 'SUBJECT_ID'), flow_ids
)

# Use the earliest possible calendar year allowed by the shifted anchor dates
anchor_group_start = candidates.ANCHOR_YEAR_GROUP.str[:4].astype(int)
years_from_anchor = candidates.INTIME.dt.year - candidates.ANCHOR_YEAR
candidates = record_cohort_step(
    cohort_flows,
    'mimiciv',
    'calendar_year_at_least_2013',
    candidates,
    candidates.loc[(anchor_group_start + years_from_anchor).ge(2013)],
    flow_ids,
)

# Select the first stay within the accepted calendar period
candidates = record_cohort_step(
    cohort_flows,
    'mimiciv',
    'first_icu_stay',
    candidates,
    candidates.sort_values(['SUBJECT_ID', 'INTIME', 'STAY_ID']).drop_duplicates('SUBJECT_ID'),
    flow_ids,
)

# Retain adults whose ICU stay lasts at least 48 hours
candidates['AGE_AT_ICU_ADMISSION'] = (
    candidates.ANCHOR_AGE + candidates.INTIME.dt.year - candidates.ANCHOR_YEAR
).clip(upper = 100)
candidates = record_cohort_step(
    cohort_flows,
    'mimiciv',
    'age_at_least_18',
    candidates,
    candidates.loc[candidates.AGE_AT_ICU_ADMISSION.ge(18)],
    flow_ids,
)
candidates = record_cohort_step(
    cohort_flows,
    'mimiciv',
    'icu_stay_at_least_48_hours',
    candidates,
    candidates.loc[candidates.LOS.ge(2)],
    flow_ids,
)

# Link admission outcomes
cohort_rows = record_cohort_step(
    cohort_flows,
    'mimiciv',
    'admission_record',
    candidates,
    candidates.merge(
        admissions[['HADM_ID', 'ADMITTIME', 'DISCHTIME', 'DEATHTIME', 'HOSPITAL_EXPIRE_FLAG']], on = 'HADM_ID'
    ),
    flow_ids,
)

# Standardise identifiers
cohort_rows['IN_HOSPITAL_MORTALITY'] = cohort_rows.HOSPITAL_EXPIRE_FLAG.astype(int)
cohort_rows['RecordID'] = cohort_rows.STAY_ID.astype(int)
cohort_rows['ICUSTAY_ID'] = cohort_rows.RecordID
cohorts['mimiciv'] = cohort_rows.sort_values('RecordID')

In [ ]:
# Extract accepted measurements and preserve raw GCS inputs for restriction experiments
for name, raw_cohort in cohorts.items():
    directory = WORK / 'data' / name
    mapping = pd.read_csv(RESOURCES / 'mappings' / f'{name}.csv')
    source = MIMICIV if name == 'mimiciv' else MIMICIII
    chart_stays = extract_mimic(source, raw_cohort, mapping, directory)

    if name == 'mimiciv':
        # Retain only stays with chart events
        raw_cohort = record_cohort_step(
            cohort_flows,
            name,
            'has_chart_events',
            raw_cohort,
            raw_cohort.loc[raw_cohort.RecordID.isin(chart_stays)].copy(),
            flow_ids,
        )

        # Append height measurements from the separate OMR table
        observations = pd.read_parquet(directory / 'observations.parquet')
        height_rows = list(mimiciv_heights(source, raw_cohort))

        if height_rows:
            observations = pd.concat([observations, *height_rows], ignore_index = True)

        write_table(directory / 'observations.parquet', observations)

    # Save the cohort with the shared column names
    stays = canonical_stays(raw_cohort, name)
    stays['partition'] = COHORT_ROLES[name]
    write_table(directory / 'cohort.parquet', stays)
    cohorts[name] = stays.set_index('stay_id')

    # Save selection counts
    flow = pd.DataFrame(cohort_flows)
    flow = flow.loc[flow.domain.eq(name)].reset_index(drop = True)
    flow.to_csv(directory / 'cohort_flow.csv', index = False)
    
    display(flow)